<!-- nav -->
[← 4 · What names refer to](04_What_Names_Refer_To.ipynb) · [Home](../../README.md) · [6 · Trees on the wire →](06_Trees_On_The_Wire.ipynb)

# 5 · One framework, three languages

**The problem.** A system's code isn't in one language. The cold-chain logger's firmware is C++, the dashboard is
TypeScript, the pipeline is Python. Tools that work on one language rarely work on another, so every refactoring, check
and generator is written three times.

mbse-programs has one framework for every language: the same trees, traversals, standards and definitions. This case
study does in C++ and TypeScript what the first four did in Python.

In [ ]:
from mbse.Programs.Ccpp import Ccpp17, Ccpp20, CcppStandard, Definitions as CppDefinitions, Syntax as S
from mbse.Programs.Framework.Syntax import walk
from mbse.Programs.Python import Python314, Syntax as P
from mbse.Programs.TypeScript import ECMAScript2025, TypeScript59, Definitions as TsDefinitions, Syntax as T
from toolkit import outline

cpp = Ccpp20.parse('''struct Reading {
    int raw;
};
bool in_range(const Reading &r) {
    return r.raw >= 200 && r.raw <= 800;
}
''')
ts = TypeScript59.parse('''interface Reading {
    celsius: number;
}
export function inRange(r: Reading): boolean {
    return r.celsius >= 2 && r.celsius <= 8;
}
''')

## Step 1: C and C++

Ccpp is C and C++ as one tree language, organized as the standard's grammar is: specifiers, declarators, statements.
The return statement's expression, in outline:

In [ ]:
function = cpp.items[1]
print(outline(function.body.items[0]))

## Step 2: TypeScript and JavaScript

TypeScript's tree follows typescript-estree, the tree ESLint reads; JavaScript is the same tree language without the
types. The same expression:

In [ ]:
print(outline(ts.body[1].declaration.body.body[0]))

## Step 3: the same tools

`walk`, `Parents`, `copy` and `Transformer` are the framework's, so they work alike on every language. Here one function
renames the parameter `r` to `reading` in all three. Each language spells an identifier its own way (`spelling` in Ccpp
and Python, `name` in TypeScript, as typescript-estree does), and that's all that differs:

In [ ]:
IDENTIFIERS = {S.Identifier: "spelling", P.Identifier: "spelling", T.Identifier: "name"}


def rename(tree, old, new):
    for node in walk(tree):
        attribute = IDENTIFIERS.get(type(node))
        if attribute and getattr(node, attribute) == old:
            setattr(node, attribute, new)
    return tree


py = Python314.parse("def in_range(r):\n    return 200 <= r.raw <= 800\n")
for standard, tree in ((Ccpp20, cpp), (TypeScript59, ts), (Python314, py)):
    print(standard.print(rename(tree, "r", "reading")))

(A rename that respects scopes uses each language's `Definitions`, as case study 4 did for Python.)

## Step 4: standards in every family

C and C++ are one tree language whose standards are both families'. C has no references and, before C23, no `bool`;
C++17 has no spaceship operator. TypeScript's own syntax isn't ECMAScript:

In [ ]:
print(CcppStandard(2011, "C").check(cpp))
print(Ccpp17.check(Ccpp20.parse("auto c = 1 <=> 2;\n")))
print(ECMAScript2025.check(ts)[:2])

## Step 5: definitions in every language

Each language's `Definitions` follows its own rules: C++'s classes and namespaces, TypeScript's values, types and
namespaces kept apart:

In [ ]:
print(list(CppDefinitions.define(cpp).entities()))
print(list(TsDefinitions.define(ts).entities()))

## Why it works this way

- **A framework, then languages.** Trees, builders, traversals, validation, standards and definitions are written once;
  a language is its kinds, its parser and printer, and its scoping rules. Adding a language adds no new tools.
- **Each language's established tree.** Python's follows `ast`, TypeScript's typescript-estree, C++'s the standard's
  grammar, so each language's programmers recognize theirs.
- **C and C++, TypeScript and JavaScript, as one each.** Their standards overlap so much that one tree language with
  two families of standards says more, and checks more, than two.

<!-- nav -->
[← 4 · What names refer to](04_What_Names_Refer_To.ipynb) · [Home](../../README.md) · [6 · Trees on the wire →](06_Trees_On_The_Wire.ipynb)